In [ ]:
%matplotlib inline


# Squidpy Visium H&E Spatial Transcriptomics Working Example

This notebook is a reproducible working example of a Squidpy spatial transcriptomics workflow. It analyzes a public 10x Genomics Visium mouse-brain H&E dataset using preprocessed `AnnData` and a matched tissue image stored as a Squidpy `ImageContainer`.

The workflow demonstrates spatial cluster visualization, image-feature extraction, morphology-based feature clustering, spatial-neighbor graph construction, neighborhood enrichment, co-occurrence, ligand-receptor analysis, and Moran's I spatial autocorrelation.

Concrete outputs are written to `output/squidpy_visium_hne/`: a processed `.h5ad` file, observation metadata, image-feature tables, spatial-statistics tables, Moran's I results, summary figures, and package-version information.

Source: official Squidpy Visium H&E example from `scverse/squidpy_notebooks`.


In [ ]:
from pathlib import Path
import importlib.metadata as importlib_metadata
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import anndata as ad
import scanpy as sc
import squidpy as sq

output_dir = Path("output") / "squidpy_visium_hne"
figures_dir = output_dir / "figures"
output_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

sc.logging.print_header()
print(f"squidpy=={sq.__version__}")

# Load the preprocessed Visium dataset and matched tissue image.
img = sq.datasets.visium_hne_image()
adata = sq.datasets.visium_hne_adata()


Visualize pre-annotated gene-expression clusters in tissue-image context with `squidpy.pl.spatial_scatter`.


In [ ]:
sq.pl.spatial_scatter(adata, color="cluster")

## Image Features

Visium datasets include high-resolution tissue images. `squidpy.im.calculate_image_features` computes morphology-derived features for each spot and stores them in `adata.obsm`, where they can be analyzed alongside gene-expression features.

This working example extracts summary image features at two spatial scales, combines them into `adata.obsm["features"]`, and clusters spots based on image morphology.


In [ ]:
# calculate features for different scales (higher value means more context)
for scale in [1.0, 2.0]:
    feature_name = f"features_summary_scale{scale}"
    sq.im.calculate_image_features(
        adata,
        img.compute(),
        features="summary",
        key_added=feature_name,
        n_jobs=4,
        scale=scale,
    )


# combine features in one dataframe
adata.obsm["features"] = pd.concat(
    [adata.obsm[f] for f in adata.obsm.keys() if "features_summary" in f],
    axis="columns",
)
# make sure that we have no duplicated feature names in the combined table
adata.obsm["features"].columns = ad.utils.make_index_unique(
    adata.obsm["features"].columns
)

Cluster the extracted image features to compare morphology-derived structure with gene-expression cluster annotations.


In [ ]:
# helper function returning a clustering
def cluster_features(features: pd.DataFrame, like=None) -> pd.Series:
    """
    Calculate leiden clustering of features.

    Specify filter of features using `like`.
    """
    # filter features
    if like is not None:
        features = features.filter(like=like)
    # create temporary adata to calculate the clustering
    adata = ad.AnnData(features)
    # important - feature values are not scaled, so need to scale them before PCA
    sc.pp.scale(adata)
    # calculate leiden clustering
    sc.pp.pca(adata, n_comps=min(10, features.shape[1] - 1))
    sc.pp.neighbors(adata)
    sc.tl.leiden(adata)

    return adata.obs["leiden"]


# calculate feature clusters
adata.obs["features_cluster"] = cluster_features(adata.obsm["features"], like="summary")

# compare feature and gene clusters
sq.pl.spatial_scatter(adata, color=["features_cluster", "cluster"])

The feature-derived clusters recapitulate some tissue structures while differing in other regions. This makes the notebook a useful working artifact for checking image/expression concordance, not just expression-only clustering.


## Spatial Statistics and Graph Analysis

Squidpy uses spatial coordinates to build a neighbor graph and compute spatial organization statistics. This section computes neighborhood enrichment for pre-annotated clusters.


In [ ]:
sq.gr.spatial_neighbors(adata)
sq.gr.nhood_enrichment(adata, cluster_key="cluster")
sq.pl.nhood_enrichment(adata, cluster_key="cluster")

Neighborhood enrichment highlights cluster pairs that occur near each other more often than expected under permutation. In this mouse-brain example, hippocampal regions show strong spatial relationships.


## Co-occurrence Across Spatial Dimensions

Co-occurrence estimates how often one cluster appears near another across increasing spatial radii. Unlike neighborhood enrichment, it uses the original spatial coordinates directly.


In [ ]:
sq.gr.co_occurrence(adata, cluster_key="cluster")
sq.pl.co_occurrence(
    adata,
    cluster_key="cluster",
    clusters="Hippocampus",
    figsize=(8, 4),
)

The co-occurrence result provides a distance-aware view of spatial relationships around the Hippocampus cluster.


## Ligand-Receptor Interaction Analysis

Squidpy can run a ligand-receptor interaction analysis over cluster pairs using CellPhoneDB-style ligand-receptor pairs extended with OmniPath annotations. Here the analysis is subset for hippocampal source and target groups.


In [ ]:
sq.gr.ligrec(
    adata,
    n_perms=100,
    cluster_key="cluster",
)
sq.pl.ligrec(
    adata,
    cluster_key="cluster",
    source_groups="Hippocampus",
    target_groups=["Pyramidal_layer", "Pyramidal_layer_dentate_gyrus"],
    means_range=(3, np.inf),
    alpha=1e-4,
    swap_axes=True,
)

The ligand-receptor dotplot identifies candidate molecular interactions that may be relevant to cellular communication in the hippocampal region. These candidates should be interpreted as hypotheses for follow-up analysis.


## Spatially Variable Genes With Moran's I

Moran's I identifies genes with spatial autocorrelation across the tissue. This example evaluates the first 1,000 highly variable genes and stores sorted results in `adata.uns["moranI"]`.


In [ ]:
genes = adata[:, adata.var.highly_variable].var_names.values[:1000]
sq.gr.spatial_autocorr(
    adata,
    mode="moran",
    genes=genes,
    n_perms=100,
    n_jobs=1,
)

The Moran's I result table is sorted by spatial autocorrelation statistic.


In [ ]:
adata.uns["moranI"].head(10)

Visualize top spatially autocorrelated genes over tissue coordinates.


In [ ]:
sq.pl.spatial_scatter(adata, color=["Olfm1", "Plp1", "Itpka", "cluster"])

The top genes localize to known mouse-brain structures such as pyramidal layers and fiber tract regions.


## Export Working-Example Artifacts


In [ ]:
# Export concrete outputs for reuse and evaluation.
adata.write_h5ad(output_dir / "squidpy_visium_hne_working_example.h5ad")

obs_columns = [
    col
    for col in [
        "cluster",
        "features_cluster",
    ]
    if col in adata.obs
]
adata.obs[obs_columns].to_csv(output_dir / "visium_hne_obs_clusters.csv")

if "features" in adata.obsm:
    adata.obsm["features"].to_csv(output_dir / "visium_hne_image_features.csv")

if "moranI" in adata.uns:
    adata.uns["moranI"].to_csv(output_dir / "visium_hne_morans_i.csv")


def export_uns_entry(key: str, prefix: str) -> None:
    value = adata.uns.get(key)
    if value is None:
        return
    if isinstance(value, pd.DataFrame):
        value.to_csv(output_dir / f"{prefix}.csv")
        return
    if isinstance(value, dict):
        for subkey, subvalue in value.items():
            out = output_dir / f"{prefix}_{subkey}.csv"
            if isinstance(subvalue, pd.DataFrame):
                subvalue.to_csv(out)
            elif hasattr(subvalue, "shape"):
                pd.DataFrame(subvalue).to_csv(out, index=False)
            else:
                (output_dir / f"{prefix}_{subkey}.txt").write_text(str(subvalue))
        return
    if hasattr(value, "shape"):
        pd.DataFrame(value).to_csv(output_dir / f"{prefix}.csv", index=False)
    else:
        (output_dir / f"{prefix}.txt").write_text(str(value))


for key in [
    "cluster_nhood_enrichment",
    "cluster_co_occurrence",
    "cluster_ligrec",
]:
    export_uns_entry(key, key)

sq.pl.spatial_scatter(
    adata,
    color=["cluster", "features_cluster"],
    show=False,
)
plt.savefig(figures_dir / "visium_hne_spatial_clusters.png", bbox_inches="tight", dpi=150)
plt.close()

sq.pl.spatial_scatter(
    adata,
    color=["Olfm1", "Plp1", "Itpka", "cluster"],
    show=False,
)
plt.savefig(figures_dir / "visium_hne_spatial_gene_expression.png", bbox_inches="tight", dpi=150)
plt.close()

version_lines = [
    f"python: {sys.version.split()[0]}",
    f"scanpy: {sc.__version__}",
    f"anndata: {ad.__version__}",
    f"squidpy: {sq.__version__}",
    f"pandas: {pd.__version__}",
]
for package in ["numpy", "scipy", "matplotlib", "skimage"]:
    try:
        version_lines.append(f"{package}: {importlib_metadata.version(package)}")
    except importlib_metadata.PackageNotFoundError:
        version_lines.append(f"{package}: not installed")

(output_dir / "package_versions.txt").write_text("\n".join(version_lines) + "\n")
